# Sentiment Analysis
The objective is to perform sentiment analysis on the Amazon Product Reviews dataset using spaCy.

In [5]:
# Import libraries
import pandas as pd
import spacy
import kagglehub
from kagglehub import KaggleDatasetAdapter
from spacytextblob.spacytextblob import SpacyTextBlob

In [15]:
def load_model(model_name):
    '''
    Load spaCy model with textblob pipeline

    Args:
        model_name (str): Name of the model to load

    Returns:
        spacy.Language: Loaded spaCy model
    '''
    try:
        nlp = spacy.load(model_name)
    except OSError:
        raise OSError(f"Error loading spaCy model '{model_name}'")

    # Add spaCyTextBlob to the pipeline
    if not nlp.has_pipe("spacytextblob"):
        try:
            nlp.add_pipe("spacytextblob")
        except Exception:
            raise ImportError("Install spacytextblob using: pip install spacytextblob")
    else:
        print("spaCyTextBlob already added to the pipeline.")

    return nlp


def load_data(file_path):
    '''
    Load the dataset from kaggle with error handling
    Set encoding to latin1 to avoid encoding errors

    Args:
        file_path (str): Path to the dataset file

    Returns:
        pd.DataFrame: Loaded dataset
    '''
    try:
        df = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "datafiniti/consumer-reviews-of-amazon-products",
            file_path,
            pandas_kwargs={"encoding": "latin1"}
        )
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f"Error, file not found: {file_path}")
    except Exception as e:
        raise Exception(f"Error loading file: {e}")


def clean_reviews(df):
    '''
    Clean the reviews.text column by removing null values and duplicates

    Args:
        df (pd.DataFrame): Reviews dataframe

    Returns:
        df_clean (pd.DataFrame): Cleaned reviews dataframe
    '''
    df_clean = df.copy()
    initial_count = len(df_clean)
    df_clean = df_clean.dropna(subset=['reviews.text'])
    df_clean = df_clean.drop_duplicates(subset=['reviews.text']).reset_index(drop=True)
    final_count = len(df_clean)
    print(f"Cleaned reviews: {initial_count} → {final_count} (removed {initial_count - final_count})")
    return df_clean


def preprocess_text(text, nlp):
    '''
    Preprocess the text by converting to lower case, strip, remove stop words and punctuation

    Args:
        text (str): Text to preprocess

    Returns:
        str: Preprocessed text
    '''
    text = str(text).lower().strip()
    doc = nlp(text)
    # Remove stop words and punctuation
    tokens = [token.text for token in doc if not token.is_stop and not token.is_punct]
    return ' '.join(tokens)  # Return cleaned text


def analyze_sentiment_blob(review, nlp):
    '''
    Analyze sentiment using TextBlob polarity via spaCy.

    Args:
        review (str): Review text
        nlp (spacy.Language): Loaded spaCy model with TextBlob pipeline

    Returns:
        tuple: (label, polarity, subjectivity)
    '''
    doc = nlp(review)

    polarity = doc._.blob.polarity  # -1 to 1
    subjectivity = doc._.blob.subjectivity  # 0 to 1

    # Determine sentiment label
    if polarity > 0.1:
        label = "Positive"
    elif polarity < -0.1:
        label = "Negative"
    else:
        label = "Neutral"

    return label, round(polarity, 4), round(subjectivity, 4)


def compare_reviews_similarity(review1, review2, nlp):
    '''
    Compare similarity between two reviews using spaCy embeddings.

    Args:
        review1 (str): First review
        review2 (str): Second review
        nlp (spacy.Language): Loaded spaCy model

    Returns:
        float: Similarity score between 0 and 1
    '''
    doc1 = nlp(review1)
    doc2 = nlp(review2)
    similarity = doc1.similarity(doc2)
    return round(similarity, 4)


def main():
    '''
    Main function to run sentiment analysis pipeline
    '''
    # Define file path for dataset and model for middle-sized English model
    file_path = "Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv"
    model_name = "en_core_web_md"

    # Load the spaCy model and dataset, then run the pipeline
    nlp = load_model(model_name)
    df = load_data(file_path)
    df = clean_reviews(df)

    # Preprocess text
    df["clean_text"] = df["reviews.text"].apply(lambda x: preprocess_text(x, nlp))

    # Test sentiment analysis on sample reviews
    sample_reviews = [
        "This product is amazing! Works better than expected.",  # Positive review
        "Terrible quality. Completely disappointed.",  # Negative review
        "The Bluetooth connection is stable, and it switches between my laptop and phone instantly."  # Neural review
    ]
    print("\n" + "-" * 60)
    print("SENTIMENT ANALYSIS RESULTS (TextBlob)")
    print("-" * 60)
    for review in sample_reviews:
        # Analyze sentiment
        label, pol, subj = analyze_sentiment_blob(review, nlp)
        print(f"Review: {review}")
        print(f"Sentiment: {label} | Polarity: {pol} | Subjectivity: {subj}\n")

    # Compare two real reviews from the dataset
    if len(df) >= 2:
        # iloc to get first two reviews
        review_a = df['reviews.text'].iloc[0]
        review_b = df['reviews.text'].iloc[1]

        # sim is similarity
        sim = compare_reviews_similarity(review_a, review_b, nlp)
        print(f"Similarity between first two reviews: {sim:.4f}")

        # Slice the first 100 characters
        print(f"Review A: {review_a[:100]}...")
        print(f"Review B: {review_b[:100]}...")

    # Add sentiment to the dataframe
    df[['sentiment_label', 'polarity', 'subjectivity']] = df['reviews.text'].apply(
        lambda x: pd.Series(analyze_sentiment_blob(x, nlp))
    )
    print("\nSample of results:")
    # Display first ten observations
    print(df[['reviews.text', 'sentiment_label', 'polarity']].head(10))

    return df, nlp


# Program starts
if __name__ == "__main__":
    df_final, nlp_model = main()

/tmp/ipython-input-296951863.py:39: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'consumer-reviews-of-amazon-products' dataset.
Cleaned reviews: 5000 → 4385 (removed 615)

------------------------------------------------------------
SENTIMENT ANALYSIS RESULTS (TextBlob)
------------------------------------------------------------
Review: This product is amazing! Works better than expected.
Sentiment: Positive | Polarity: 0.3833 | Subjectivity: 0.6

Review: Terrible quality. Completely disappointed.
Sentiment: Negative | Polarity: -0.875 | Subjectivity: 0.875

Review: The Bluetooth connection is stable, and it switches between my laptop and phone instantly.
Sentiment: Neutral | Polarity: 0.0 | Subjectivity: 0.6667

Similarity between first two reviews: 0.8651
Review A: I thought it would be as big as small paper but turn out to be just like my palm. I think it is too ...
Review B: This kindle is light and easy to use especially at the beach!!!...

Sample of results:
                                        reviews.text senti